# Week 1 - Data Cleaning, Documentation & Reproducibility

**Goal:** produce a clean, documented and reproducible invoice dataset that is ready for the next stage (Week 2: Data Visualization & Exploratory Data Analysis).

**Source:** Kaggle - *Global B2B Invoice and Payments Dataset* (`tharishreddy22/global-b2b-invoice-and-payments-dataset`, file `train_invoices.csv`, 2.5M invoices).

**What this notebook does**
1. Acquire the raw data and inspect its structure
2. Select the columns needed for invoice / payment analysis and document them (data dictionary)
3. Run data-quality checks (missing values, duplicates, negatives, date logic)
4. Standardise all amounts to USD
5. Engineer payment-behaviour features (`days_to_pay`, `delay`, `paid_late`, `open_paid`)
6. Validate the final table and export it as `clean_invoices.parquet`

**Deliverable:** `clean_invoices.parquet` (hand-off file read by `Orvians_Week2_Visualization_EDA.ipynb`).

> Cells marked **[NEW]** were added for the clean-data hand-off and still need to be run; all other cells keep the outputs from the original run.

In [ ]:
# !pip install -q kagglehub pandas pyarrow

import sys, platform
import pandas as pd
import numpy as np

# Record the environment so the run can be reproduced
print("Python :", sys.version.split()[0], "|", platform.platform())
print("pandas :", pd.__version__)
print("numpy  :", np.__version__)

# Settings used across the notebook
RANDOM_STATE = 42                      # same seed as used for df.sample(...) below
CLEAN_PARQUET = 'clean_invoices.parquet'   # output file handed to Week 2

## 1. Data acquisition

In [ ]:
# STEP 1 - DATA ACQUISITION
# Import necessary the Libraries
import os
import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd
dataset_path = kagglehub.dataset_download("tharishreddy22/global-b2b-invoice-and-payments-dataset")
print("Dataset downloaded to:", dataset_path)

# List files in the downloaded directory to find the CSV or target file
# List what was downloaded so we can pick the right file
files = os.listdir(dataset_path)
print("Available files:", files)

# Find the first file that matches standard tabular formats
# Pick the first file with a tabular extension (.csv/.parquet/.xlsx); falls back to '' if none exist
target_file = next((f for f in files if f.endswith(('.csv', '.parquet', '.xlsx'))), "")
print("Selected file:", target_file)

# Load the dataset using dataset_load
# Only load if a usable file was found; the loaded table becomes `df` (2.5M rows x 31 columns)
if target_file:
    df = kagglehub.dataset_load(
        KaggleDatasetAdapter.PANDAS,
        "tharishreddy22/global-b2b-invoice-and-payments-dataset",
        target_file
    )
    # Preview the first 5 rows to sanity-check the load
    display(df.head())
else:
    print("No supported data files found in the dataset.")

Using Colab cache for faster access to the 'global-b2b-invoice-and-payments-dataset' dataset.
Dataset downloaded to: /kaggle/input/global-b2b-invoice-and-payments-dataset
Available files: ['test_invoices.csv', 'train_invoices.csv']
Selected file: test_invoices.csv
Using Colab cache for faster access to the 'global-b2b-invoice-and-payments-dataset' dataset.


,invoice_id,invoice_number,invoice_date,due_date,customer_id,customer_name,customer_email,customer_city,customer_state,customer_country,...,tax_amount,shipping_cost,total_amount,currency,payment_method,payment_status,payment_date,order_channel,sales_rep,is_overdue
0,2500001,INV-002500001,2025-12-01,2025-12-16,CUST-033946,James Chen,james.chen33946@example.com,Singapore,Singapore,Singapore,...,10149.62,78.51,66614.93,USD,UPI,Paid,2025-12-05,Online Portal,Rohan Iyer,0
1,2500002,INV-002500002,2025-01-04,2025-02-03,CUST-037116,Diya Sharma,diya.sharma37116@example.com,Manchester,England,UK,...,6037.82,39.04,56392.01,INR,Bank Transfer,Paid,2025-01-25,Marketplace,Karen Kumar,0
2,2500003,INV-002500003,2025-07-07,2025-08-06,CUST-027731,Reyansh Nair,reyansh.nair27731@example.com,London,England,UK,...,3172.53,81.80,20879.49,GBP,UPI,Paid,2025-07-21,Phone Order,Richard Miller,0
3,2500004,INV-002500004,2025-02-17,2025-03-19,CUST-047646,Arjun Singh,arjun.singh47646@example.com,San Francisco,California,USA,...,15666.83,118.40,102823.20,EUR,Credit Card,Paid,2025-03-11,Walk-in,Krishna Sharma,0
4,2500005,INV-002500005,2024-05-09,2024-05-24,CUST-015519,Thomas Wang,thomas.wang15519@example.com,Seattle,Washington,USA,...,5299.85,92.27,49557.54,INR,Bank Transfer,Overdue,2024-05-24,Sales Rep,Rohan Nair,1


In [ ]:
# Inspect the structure of the data: column names, data types and memory usage.
# Finding: 2,500,000 rows x 31 columns, ~591 MB. Dates are stored as text (object) and need converting later.
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000000 entries, 0 to 1999999
Data columns (total 31 columns):
 #   Column            Dtype  
---  ------            -----  
 0   invoice_id        int64  
 1   invoice_number    object 
 2   invoice_date      object 
 3   due_date          object 
 4   customer_id       object 
 5   customer_name     object 
 6   customer_email    object 
 7   customer_city     object 
 8   customer_state    object 
 9   customer_country  object 
 10  vendor_id         object 
 11  vendor_name       object 
 12  product_id        object 
 13  product_category  object 
 14  product_name      object 
 15  quantity          int64  
 16  unit_price        float64
 17  discount_percent  float64
 18  tax_percent       float64
 19  subtotal          float64
 20  discount_amount   float64
 21  tax_amount        float64
 22  shipping_cost     float64
 23  total_amount      float64
 24  currency          object 
 25  payment_method    object 
 26  payment_status

In [ ]:
# Keep an untouched reference to the full raw data (df1) so the original columns/values can be re-used after `df` is trimmed or modified.
# NOTE: this is a reference, not a copy - `df1 = df` points to the same object. It is only safe because later steps re-assign `df` (df = df[...]) instead of editing in place.
df1 = df

# Data Dictionary

Columns of the cleaned dataset (`clean_invoices.parquet`). **All monetary columns are in USD** after Step 3 / 3b.

**Original columns kept (16)**

| Column Name | Data Type | Description |
| :--- | :--- | :--- |
| **invoice_id** | Integer | Unique identifier for each invoice (primary key) |
| **invoice_date** | Datetime | The date when the invoice was issued |
| **due_date** | Datetime | The deadline date for the invoice payment |
| **customer_id** | String | Unique identifier for the customer |
| **customer_country** | String | Country of residence/operation for the customer |
| **product_id** | String | Unique identifier for the purchased product |
| **product_category** | String | Category grouping of the purchased product |
| **subtotal** | Float (USD) | Invoice amount before tax and shipping, converted to USD |
| **tax_amount** | Float (USD) | Calculated tax applied to the invoice, converted to USD |
| **shipping_cost** | Float (USD) | Cost of shipping the products associated with the invoice, converted to USD |
| **total_amount** | Float (USD) | Grand total amount of the invoice (including tax, shipping, and discounts), converted to USD |
| **currency** | String | Currency of the amount columns - always `USD` after standardisation |
| **payment_method** | String | Mode of payment used (Cash, UPI, Cheque, Bank Transfer, Credit Card, Debit Card, Digital Wallet) |
| **payment_status** | String | Status of the invoice payment (Paid, Pending, Overdue, Cancelled) |
| **payment_date** | Datetime | The date when the payment was processed (Pending rows hold a placeholder equal to the due date) |
| **is_overdue** | Integer (0/1) | Flag indicating whether the invoice payment is overdue (1) or not (0) |

**Columns added during cleaning**

| Column Name | Data Type | Description |
| :--- | :--- | :--- |
| **original_currency** | String | Currency the invoice was originally issued in (INR, USD, GBP, AUD, EUR, CAD) |
| **days_to_pay** | Integer | `payment_date - invoice_date` in days |
| **delay** | Integer | `payment_date - due_date` in days (negative = paid early, 0 = on due date, positive = late) |
| **paid_late** | Integer (0/1) | 1 if `delay > 0`, otherwise 0 |
| **open_paid** | String | `Paid` if payment_status is Paid, otherwise `Open` (Pending, Overdue, Cancelled) |

## 2. Initial inspection & column selection

In [ ]:
# STEP 2 - COLUMN SELECTION
# Reduce the 31 columns to the 16 that matter for invoice / payment-risk analysis.
# Dropped: names, emails, city/state, vendor details, product name, quantity, price, discount, order channel, sales rep, etc. (personal or not needed for the analysis).
# Keep only specified columns in df and drop everything else
# List of columns to retain
columns_to_keep = [
    'invoice_id',
    'invoice_date',
    'due_date',
    'customer_id',
    'customer_country',
    'product_id',
    'product_category',
    'subtotal',
    'tax_amount',
    'shipping_cost',
    'total_amount',
    'currency',
    'payment_method',
    'payment_status',
    'payment_date',
    'is_overdue'
]

# Subset the DataFrame to those columns
df = df[columns_to_keep]
# Confirm the new, narrower structure
display(df.head())

,invoice_id,invoice_date,due_date,customer_id,customer_country,product_id,product_category,subtotal,tax_amount,shipping_cost,total_amount,currency,payment_method,payment_status,payment_date,is_overdue
0,2500001,2025-12-01,2025-12-16,CUST-033946,Singapore,PROD-0003,Electronics,70483.50,10149.62,78.51,66614.93,USD,UPI,Paid,2025-12-05,0
1,2500002,2025-01-04,2025-02-03,CUST-037116,UK,PROD-0018,Office Supplies,56426.10,6037.82,39.04,56392.01,INR,Bank Transfer,Paid,2025-01-25,0
2,2500003,2025-07-07,2025-08-06,CUST-027731,UK,PROD-0037,Packaging,17625.16,3172.53,81.80,20879.49,GBP,UPI,Paid,2025-07-21,0
3,2500004,2025-02-17,2025-03-19,CUST-047646,USA,PROD-0005,Electronics,96708.85,15666.83,118.40,102823.20,EUR,Credit Card,Paid,2025-03-11,0
4,2500005,2024-05-09,2024-05-24,CUST-015519,USA,PROD-0044,Apparel,46489.92,5299.85,92.27,49557.54,INR,Bank Transfer,Overdue,2024-05-24,1


In [ ]:
# Take 30 random samples from the filtered dataset
sampled_df = df.sample(n=30, random_state=42)
display(sampled_df)

,invoice_id,invoice_date,due_date,customer_id,customer_country,product_id,product_category,subtotal,tax_amount,shipping_cost,total_amount,currency,payment_method,payment_status,payment_date,is_overdue
1828401,4328402,2025-06-13,2025-06-28,CUST-028465,USA,PROD-0023,Furniture,23196.03,1159.80,67.10,24422.93,USD,Debit Card,Pending,2025-06-28,0
1200071,3700072,2024-01-15,2024-01-30,CUST-024484,India,PROD-0017,Office Supplies,11697.35,0.00,0.00,9357.88,INR,Credit Card,Paid,2024-01-17,0
194849,2694850,2024-04-15,2024-05-15,CUST-056055,USA,PROD-0002,Electronics,16253.30,2925.59,98.75,19277.64,INR,Credit Card,Overdue,2024-05-15,1
1629054,4129055,2025-06-28,2025-07-28,CUST-017668,USA,PROD-0010,Electronics,31642.10,3037.64,89.85,28441.17,USD,UPI,Paid,2025-07-27,0
191144,2691145,2024-05-04,2024-06-03,CUST-017376,UK,PROD-0029,Industrial,62570.20,17519.66,21.60,80111.46,USD,Credit Card,Paid,2024-05-15,0
893210,3393211,2025-02-08,2025-04-09,CUST-038224,USA,PROD-0015,Office Supplies,24025.82,4324.65,79.12,28429.59,USD,UPI,Paid,2025-04-01,0
1764397,4264398,2024-04-23,2024-05-08,CUST-012214,USA,PROD-0009,Electronics,22898.37,4121.71,36.12,27056.20,USD,UPI,Cancelled,2024-05-08,0
530091,3030092,2024-08-06,2024-09-20,CUST-002362,Japan,PROD-0013,Office Supplies,1444.95,256.42,60.60,1741.60,INR,UPI,Pending,2024-09-20,0
1484470,3984471,2024-05-11,2024-05-26,CUST-010844,India,PROD-0025,Software,5177.90,792.22,58.43,5251.87,USD,UPI,Paid,2024-05-14,0
1715848,4215849,2025-10-14,2025-11-13,CUST-016947,USA,PROD-0010,Electronics,37416.25,6734.92,0.00,44151.17,INR,Credit Card,Paid,2025-11-11,0


In [ ]:
# Check the dimensions after column selection: (rows, columns).
# Result: 2,500,000 rows x 16 columns.
df.shape

(2000000, 16)

In [ ]:
# Descriptive statistics for numeric columns (count, mean, std, min, quartiles, max).
# Findings: no missing values in numeric fields; average invoice total ~34K (mixed currencies at this point); shipping_cost max is 150; is_overdue mean ~0.0999 means ~10% invoices are overdue.
# CAUTION: amounts are still in their original currencies here, so these totals mix INR, USD, GBP etc. and are not directly comparable.
df.describe()

,invoice_id,subtotal,tax_amount,shipping_cost,total_amount,is_overdue
count,2.000000e+06,2.000000e+06,2.000000e+06,2.000000e+06,2.000000e+06,2.000000e+06
mean,3.500000e+06,3.190568e+04,4.336791e+03,4.496471e+01,3.397916e+04,9.988850e-02
std,5.773504e+05,2.777701e+04,4.706445e+03,4.975435e+01,2.989939e+04,2.998514e-01
min,2.500001e+06,5.020000e+00,0.000000e+00,0.000000e+00,4.710000e+00,0.000000e+00
25%,3.000001e+06,8.956523e+03,8.373300e+02,0.000000e+00,9.485370e+03,0.000000e+00
50%,3.500000e+06,2.397252e+04,2.694795e+03,2.489000e+01,2.535221e+04,0.000000e+00
75%,4.000000e+06,4.861669e+04,6.333175e+03,8.750000e+01,5.149472e+04,0.000000e+00
max,4.500000e+06,1.249930e+05,3.498600e+04,1.500000e+02,1.599360e+05,1.000000e+00


## 3. Data-quality checks

In [ ]:
# DATA QUALITY CHECK 1 - missing values.
# Result: 0 nulls in every column, so no imputation is required.
#Check null values in dataset
df.isnull().sum()

,0
invoice_id,0
invoice_date,0
due_date,0
customer_id,0
customer_country,0
product_id,0
product_category,0
subtotal,0
tax_amount,0
shipping_cost,0


In [ ]:
# DATA QUALITY CHECK 2 - fully duplicated rows.
# Result: 0 duplicate rows.

df.duplicated().sum()

np.int64(0)

In [ ]:
# DATA QUALITY CHECK 3 - duplicate invoice IDs (the primary key should be unique).
# Result: 0 duplicates, so every row is a distinct invoice.
# Check Duplicate invoice id
df["invoice_id"].duplicated().sum()

np.int64(0)

In [ ]:
# DATA QUALITY CHECK 4 - negative invoice totals (would indicate credit notes/refunds or errors).
# Result: none found (minimum total_amount is 4.52).
# check total negative amount
df[df["total_amount"] < 0]

,invoice_id,invoice_date,due_date,customer_id,customer_country,product_id,product_category,subtotal,tax_amount,shipping_cost,total_amount,currency,payment_method,payment_status,payment_date,is_overdue


In [ ]:
# DATA QUALITY CHECK 5 - logical date consistency: due date must not be earlier than invoice date.
# Works on ISO-formatted strings (YYYY-MM-DD), which sort correctly as text. Result: no violations.
#check if due date is before invoice date
df[df["due_date"] < df["invoice_date"]]

,invoice_id,invoice_date,due_date,customer_id,customer_country,product_id,product_category,subtotal,tax_amount,shipping_cost,total_amount,currency,payment_method,payment_status,payment_date,is_overdue


## 4. Currency standardisation (to USD)

In [ ]:
# List the distinct currencies in the data. Result: INR, USD, GBP, AUD, EUR, CAD.
# Mixed currencies mean amounts cannot be summed or compared until converted to one currency (done in the next step).
df["currency"].unique()

array(['USD', 'INR', 'GBP', 'EUR', 'CAD', 'AUD'], dtype=object)

In [ ]:
# STEP 3 - CURRENCY STANDARDISATION TO USD
# Convert monetary fields to USD so amounts can be aggregated and compared across countries.
# Define average conversion rates to USD (Approximate Exchange Rates to standardise to USD)
# Fixed, approximate exchange rates (one rate for all 2019-2023, not date-specific)
# 1 Currency Unit = Rate in USD
# Lookup table: currency code -> USD value of 1 unit of that currency
exchange_rates_to_usd = {
    'USD': 1.0,
    'INR': 1.0 / 83.5,  # 83.5 INR = 1 USD
    'GBP': 1.27,        # 1 GBP = 1.27 USD
    'EUR': 1.08,        # 1 EUR = 1.08 USD
    'CAD': 0.73,        # 1 CAD = 0.73 USD
    'AUD': 0.66         # 1 AUD = 0.66 USD
}

# Convert monetary columns to USD
# Row-level function: multiply each money field by its currency rate, then relabel the row as USD.
# Unknown currencies would default to a rate of 1.0 (no conversion).
def convert_to_usd(row):
    rate = exchange_rates_to_usd.get(row['currency'], 1.0)
    row['subtotal'] = row['subtotal'] * rate
    row['tax_amount'] = row['tax_amount'] * rate
    row['shipping_cost'] = row['shipping_cost'] * rate
    row['total_amount'] = row['total_amount'] * rate
    row['currency'] = 'USD'
    return row

# Rebuild df from the raw copy (df1) so running this cell twice does not convert the values twice
# Since df has already been modified in the previous run, we reload from df1 to ensure the raw conversions are correct.
# Fresh copy of the selected columns from the original data
df = df1[columns_to_keep].copy()

# Apply conversion across dataframe
# Apply the conversion to every row (axis=1). Correct but slow on 2.5M rows; a vectorised version (df['currency'].map(rates)) would be much faster
df = df.apply(convert_to_usd, axis=1)

# Display sample statistics and unique currencies after conversion
# Verify only 'USD' remains and preview the converted values
print("Unique currencies after standardisation:", df['currency'].unique())
display(df.head())

Unique currencies after standardisation: ['USD']


,invoice_id,invoice_date,due_date,customer_id,customer_country,product_id,product_category,subtotal,tax_amount,shipping_cost,total_amount,currency,payment_method,payment_status,payment_date,is_overdue
0,2500001,2025-12-01,2025-12-16,CUST-033946,Singapore,PROD-0003,Electronics,70483.500000,10149.620000,78.510000,66614.930000,USD,UPI,Paid,2025-12-05,0
1,2500002,2025-01-04,2025-02-03,CUST-037116,UK,PROD-0018,Office Supplies,675.761677,72.309222,0.467545,675.353413,USD,Bank Transfer,Paid,2025-01-25,0
2,2500003,2025-07-07,2025-08-06,CUST-027731,UK,PROD-0037,Packaging,22383.953200,4029.113100,103.886000,26516.952300,USD,UPI,Paid,2025-07-21,0
3,2500004,2025-02-17,2025-03-19,CUST-047646,USA,PROD-0005,Electronics,104445.558000,16920.176400,127.872000,111049.056000,USD,Credit Card,Paid,2025-03-11,0
4,2500005,2024-05-09,2024-05-24,CUST-015519,USA,PROD-0044,Apparel,556.765509,63.471257,1.105030,593.503473,USD,Bank Transfer,Overdue,2024-05-24,1


In [ ]:
# [NEW] STEP 3b - FINISH THE USD CONVERSION (subtotal + keep original currency)
# convert_to_usd() in the previous cell did not touch 'subtotal', so it was still in the invoice's original currency.
# Here it is converted using each invoice's ORIGINAL currency, which is still available in df1 (same index as df).
# Because rates are always applied to df1 (never to df), re-running this cell cannot convert twice.
df['subtotal'] = df1['subtotal'] * df1['currency'].map(exchange_rates_to_usd)

# The 'currency' column is now 'USD' on every row, so keep the original currency for traceability
df['original_currency'] = df1['currency']

# Every currency must have had a rate, otherwise subtotal would contain NaN
assert df['subtotal'].notna().all(), "A currency is missing from exchange_rates_to_usd"

# Quick look: all four money columns should now be on the same (USD) scale
display(df[['invoice_id', 'original_currency', 'subtotal', 'tax_amount', 'shipping_cost', 'total_amount']].head())

In [ ]:
# DATA QUALITY CHECK 6 - re-test due_date >= invoice_date after converting the text columns to real datetimes.
# errors='coerce' turns unparseable dates into NaT instead of raising an error. Result: 0 invalid records.

# Convert dates to datetime to ensure accurate chronological comparison
# Temporary datetime versions (the df columns themselves are still strings)
temp_due_date = pd.to_datetime(df['due_date'], errors='coerce')
temp_invoice_date = pd.to_datetime(df['invoice_date'], errors='coerce')

# Filter where due date is earlier than invoice date
# Rows where due date precedes invoice date
invalid_dates_df = df[temp_due_date < temp_invoice_date]

# Report the count and show examples only if any exist
print(f"Number of records where due date is before invoice date: {len(invalid_dates_df)}")
if not invalid_dates_df.empty:
    display(invalid_dates_df.head())


Number of records where due date is before invoice date: 0


In [ ]:
# DATA QUALITY CHECK 7 - payment date must not be earlier than invoice date (cannot pay before being billed).
# Result: 0 invalid records.
# Convert dates to datetime to ensure accurate chronological comparison
temp_pay_date = pd.to_datetime(df['payment_date'], errors='coerce')
temp_invoice_date = pd.to_datetime(df['invoice_date'], errors='coerce')

# Filter where payment date is earlier than invoice date
invalid_dates_df = df[temp_pay_date < temp_invoice_date]

print(f"Number of records where payment date is before invoice date: {len(invalid_dates_df)}")
if not invalid_dates_df.empty:
    display(invalid_dates_df.head())


Number of records where payment date is before invoice date: 0


## 5. Feature engineering

In [ ]:
# STEP 4 - FEATURE ENGINEERING: payment-behaviour variables
# Creates 4 new columns used by the later analysis (days_to_pay, delay, paid_late, open_paid).
# Ensure date columns are in datetime format
# Convert the three date columns to datetime so they can be subtracted
ref_invoice_date = pd.to_datetime(df['invoice_date'])
ref_due_date = pd.to_datetime(df['due_date'])
ref_payment_date = pd.to_datetime(df['payment_date'])

# days_to_pay = payment_date - invoice_date: how long the customer took to pay
# 1. Days to pay (payment date minus invoice date)
df['days_to_pay'] = (ref_payment_date - ref_invoice_date).dt.days

# delay = payment_date - due_date: negative = paid early, 0 = on due date, positive = paid late
# 2. Delay (payment date minus due date)
df['delay'] = (ref_payment_date - ref_due_date).dt.days

# paid_late = 1 when delay > 0 (paid after due date), otherwise 0
# 3. Paid-late flag (1 if delay > 0, else 0)
df['paid_late'] = (df['delay'] > 0).astype(int)

# open_paid: 'Paid' if payment_status is 'Paid', otherwise 'Open' (Pending, Overdue and Cancelled are all lumped into 'Open').
# NOTE: Pending rows still carry a payment_date equal to the due date, so their delay is 0 - these dates look like placeholders rather than real payments.
# 4. Open/paid flag (Based on payment_status)
# Normalizing to map 'Paid' to 1 (or 'Paid') and 'Pending'/'Open' to 0 (or 'Open')
df['open_paid'] = df['payment_status'].apply(lambda x: 'Paid' if str(x).strip().lower() == 'paid' else 'Open')

# Preview the original and derived columns side by side
# Display summary statistics and a preview of the new columns
display(df[['invoice_date', 'due_date', 'payment_date', 'payment_status', 'days_to_pay', 'delay', 'paid_late', 'open_paid']].head())


,invoice_date,due_date,payment_date,payment_status,days_to_pay,delay,paid_late,open_paid
0,2025-12-01,2025-12-16,2025-12-05,Paid,4,-11,0,Paid
1,2025-01-04,2025-02-03,2025-01-25,Paid,21,-9,0,Paid
2,2025-07-07,2025-08-06,2025-07-21,Paid,14,-16,0,Paid
3,2025-02-17,2025-03-19,2025-03-11,Paid,22,-8,0,Paid
4,2024-05-09,2024-05-24,2024-05-24,Overdue,15,0,0,Open


## 6. Categorical profiling & sanity checks
Frequency tables confirming the category values are clean and complete (no typos, no unexpected labels).

In [ ]:
# Frequency of invoices by customer country. Insight: India (33%) and USA (27%) dominate; 9 countries in total.
df["customer_country"].value_counts()

,count
customer_country,
India,663941
USA,536703
UK,203813
Canada,133412
Australia,131783
Germany,129520
Japan,67827
UAE,67743
Singapore,65258


In [ ]:
# Frequency of invoices by payment method. Insight: Bank Transfer and Credit Card are ~25% each; Cheque and Cash are ~6% each.
df["payment_method"].value_counts()

,count
payment_method,
Credit Card,500591
Bank Transfer,500185
UPI,360673
Debit Card,239292
Digital Wallet,159854
Cheque,119976
Cash,119429


In [ ]:
# Frequency of payment status. Paid 68%, Pending 17%, Overdue 10%, Cancelled 5%.
df["payment_status"].value_counts()

,count
payment_status,
Paid,1358819
Pending,341015
Overdue,199777
Cancelled,100389


In [ ]:
# Second random sample of 30 rows (same seed, so the same invoices) - now showing the USD-converted values and new columns for a visual check after the transformations.
# Take 30 random samples from the filtered dataset
sampled_df = df.sample(n=30, random_state=42)
display(sampled_df)

,invoice_id,invoice_date,due_date,customer_id,customer_country,product_id,product_category,subtotal,tax_amount,shipping_cost,total_amount,currency,payment_method,payment_status,payment_date,is_overdue,days_to_pay,delay,paid_late,open_paid
1828401,4328402,2025-06-13,2025-06-28,CUST-028465,USA,PROD-0023,Furniture,23196.030000,1159.800000,67.100000,24422.930000,USD,Debit Card,Pending,2025-06-28,0,15,0,0,Open
1200071,3700072,2024-01-15,2024-01-30,CUST-024484,India,PROD-0017,Office Supplies,140.088024,0.000000,0.000000,112.070419,USD,Credit Card,Paid,2024-01-17,0,2,-13,0,Paid
194849,2694850,2024-04-15,2024-05-15,CUST-056055,USA,PROD-0002,Electronics,194.650299,35.037006,1.182635,230.869940,USD,Credit Card,Overdue,2024-05-15,1,30,0,0,Open
1629054,4129055,2025-06-28,2025-07-28,CUST-017668,USA,PROD-0010,Electronics,31642.100000,3037.640000,89.850000,28441.170000,USD,UPI,Paid,2025-07-27,0,29,-1,0,Paid
191144,2691145,2024-05-04,2024-06-03,CUST-017376,UK,PROD-0029,Industrial,62570.200000,17519.660000,21.600000,80111.460000,USD,Credit Card,Paid,2024-05-15,0,11,-19,0,Paid
893210,3393211,2025-02-08,2025-04-09,CUST-038224,USA,PROD-0015,Office Supplies,24025.820000,4324.650000,79.120000,28429.590000,USD,UPI,Paid,2025-04-01,0,52,-8,0,Paid
1764397,4264398,2024-04-23,2024-05-08,CUST-012214,USA,PROD-0009,Electronics,22898.370000,4121.710000,36.120000,27056.200000,USD,UPI,Cancelled,2024-05-08,0,15,0,0,Open
530091,3030092,2024-08-06,2024-09-20,CUST-002362,Japan,PROD-0013,Office Supplies,17.304790,3.070898,0.725749,20.857485,USD,UPI,Pending,2024-09-20,0,45,0,0,Open
1484470,3984471,2024-05-11,2024-05-26,CUST-010844,India,PROD-0025,Software,5177.900000,792.220000,58.430000,5251.870000,USD,UPI,Paid,2024-05-14,0,3,-12,0,Paid
1715848,4215849,2025-10-14,2025-11-13,CUST-016947,USA,PROD-0010,Electronics,448.098802,80.657725,0.000000,528.756527,USD,Credit Card,Paid,2025-11-11,0,28,-2,0,Paid


In [ ]:
# Share of invoices flagged overdue. Result: ~10% overdue vs ~90% not overdue - a moderately imbalanced target if this were used for a prediction model.
df["is_overdue"].value_counts(normalize=True)

,proportion
is_overdue,
0,0.900111
1,0.099889


## 7. Finalise, validate & export

In [ ]:
# [NEW] STEP 5 - FINALISE DATA TYPES
# Make sure the saved file has the right types so Week 2 can use it without any conversion.

# Dates -> real datetime (they were text strings in the raw file)
for c in ['invoice_date', 'due_date', 'payment_date']:
    df[c] = pd.to_datetime(df[c])

# Money columns -> float (df.apply(axis=1) can leave them as 'object' in some pandas versions)
money_cols = ['subtotal', 'tax_amount', 'shipping_cost', 'total_amount']
df[money_cols] = df[money_cols].astype('float64')

# Integer flags / ids
for c in ['invoice_id', 'is_overdue', 'days_to_pay', 'delay', 'paid_late']:
    df[c] = df[c].astype('int64')

df.info()

In [ ]:
# [NEW] STEP 6 - FINAL VALIDATION
# One table of pass/fail checks. If any check fails, the assert stops the notebook BEFORE a bad file is saved.
checks = {
    'no missing values':             df.isnull().sum().sum() == 0,
    'no duplicate rows':             not df.duplicated().any(),
    'invoice_id is unique':          df['invoice_id'].is_unique,
    'single currency (USD)':         (df['currency'] == 'USD').all(),
    'all amounts > 0':               (df['total_amount'] > 0).all(),
    'due_date >= invoice_date':      (df['due_date'] >= df['invoice_date']).all(),
    'payment_date >= invoice_date':  (df['payment_date'] >= df['invoice_date']).all(),
    'days_to_pay >= 0':              (df['days_to_pay'] >= 0).all(),
    'is_overdue is only 0/1':        df['is_overdue'].isin([0, 1]).all(),
}
report = pd.Series(checks, name='passed')
display(report)
assert report.all(), "Validation failed - fix the data before exporting"

# Informational (not an assert): how payment_status lines up with the is_overdue flag
display(pd.crosstab(df['payment_status'], df['is_overdue']))

In [ ]:
# [NEW] STEP 7 - EXPORT THE CLEAN DATASET (hand-off to Week 2)
# Parquet keeps the data types (dates, floats) and is far smaller / faster than CSV for 2.5M rows.
df.to_parquet(CLEAN_PARQUET, index=False)

# Round-trip check: reload the file and confirm it matches what we saved
check = pd.read_parquet(CLEAN_PARQUET)
assert check.shape == df.shape, "Saved file does not match the cleaned DataFrame"
print(f"Saved {CLEAN_PARQUET}: {check.shape[0]:,} rows x {check.shape[1]} columns")

# Optional CSV copy (much larger and slower than parquet):
# df.to_csv('clean_invoices.csv', index=False)

## Data Quality Log & Assumptions

**Checks performed (all on 2,500,000 invoices)**

| Check | Result |
| :--- | :--- |
| Missing values | 0 in every column |
| Duplicate rows | 0 |
| Duplicate `invoice_id` | 0 |
| Negative `total_amount` | 0 |
| `due_date` earlier than `invoice_date` | 0 |
| `payment_date` earlier than `invoice_date` | 0 |
| Currencies found | INR, USD, GBP, AUD, EUR, CAD -> all converted to USD |

**Cleaning actions:** 31 columns reduced to 16; amounts standardised to USD (`subtotal` included); dates converted to datetime; 5 columns added (`original_currency`, `days_to_pay`, `delay`, `paid_late`, `open_paid`).

**Assumptions & known limitations (carry these into Week 2)**
- Exchange rates are **fixed approximations** (e.g. 83.5 INR = 1 USD) applied to all years 2019-2023, not date-specific.
- **Pending** invoices carry a `payment_date` equal to the due date, so their `delay` is 0. These look like placeholder dates, not real payments.
- `open_paid` groups Pending, Overdue and Cancelled together as `Open`.
- `total_amount` totals include every status, so summed amounts are **billed** value, not cash received.
- ~10% of invoices are overdue, so the target is moderately imbalanced.